In [1]:
# import grequests
import pandas as pd
import datasets
# from utils import compile_and_test
import base64

from genai.utils import read_prompt
# from genai.Ollama import OllamaAPI

def create_index(ds, index_name):
    index_c = {}

    for idx, problem_id in enumerate(ds[index_name]):
        if problem_id not in index_c:
            index_c[problem_id] = []

        index_c[problem_id].append(idx)
    
    return index_c


dataset_submissions = pd.read_csv('data/dataset.csv')
dataset_submissions = dataset_submissions.set_index('submissions_id')
dataset_submissions['total_code_mod'] = dataset_submissions['code_additions'] + dataset_submissions['code_deletions']

dataset_submissions = dataset_submissions.sort_values('total_code_mod')

generated_tests_ds = datasets.load_dataset("data/codeforces/generated_tests")['test']

generated_tests_ds_index_mapping = create_index(generated_tests_ds, 'problem_id')

all_contests = set([k.split('/')[0] for k in generated_tests_ds_index_mapping.keys()])

problems_train = datasets.load_dataset("data/codeforces/data")['train']
problems_test = datasets.load_dataset("data/codeforces/data")['test']

problems_train_ds_index_mapping = create_index(problems_train, 'id')
problems_test_ds_index_mapping = create_index(problems_test, 'id')

/home/vscode/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import re
re_code1 = re.compile(r'```c\+\+(.+)```')
re_code2 = re.compile(r'```cpp(.+)```')
re_code3 = re.compile(r'```c(.+)```')
re_code4 = re.compile(r'```(.+)```')
think_code4 = re.compile((r'```<think>(.+)</think>```'))

def extract_code(s):
    s = s.replace('\n', '\\\\n')
    s = think_code4.sub('', s)
    found_code = re_code1.findall(s)
    if len(found_code) == 0:
        found_code = re_code2.findall(s)
      
    if len(found_code) == 0:
        found_code = re_code3.findall(s)
    
    if len(found_code) == 0:
        found_code = re_code4.findall(s)

    return found_code[0].replace('\\\\n', '\n')

print(extract_code('''```c++#include<iostream>
using namespace std;

int main()
{
  return 0;
}```'''))
    

#include<iostream>
using namespace std;

int main()
{
  return 0;
}


In [ ]:
import os
import datetime;
from utils import compile_and_test
from genai.endpoints import OpenAIAPI, OllamaAPI
import json
import dotenv
from tqdm import tqdm

dotenv.load_dotenv(dotenv_path='.devcontainer/.env', override=True)

VENDOR = 'ollama'
MODEL_NAME = 'deepseek-r1:1.5b'

if VENDOR == 'openai':
    model_api = OpenAIAPI(os.environ['OPENAI_ENDPOINT'], os.environ['OPENAI_TOKEN'])
else:
    model_api = OllamaAPI(os.environ['OLLAMA_ENDPOINT'], os.environ['OLLAMA_TOKEN'])

MAX_TRIES = 3

for _, s in tqdm(list(dataset_submissions.iterrows())):
    if s['AcceptedAnchor'] == -1 or s['verdict'] !='WRONG_ANSWER':
        continue

    if s['problem_id'] in problems_train_ds_index_mapping:
        problem_data = problems_train[problems_train_ds_index_mapping[s['problem_id']][0]]
    elif s['problem_id'] in problems_test_ds_index_mapping:
        problem_data = problems_test[problems_test_ds_index_mapping[s['problem_id']][0]]
    else:
        continue
    
    programmingLanguage = s['programmingLanguage']
    source_code = base64.b64decode(s['sourceCode']).decode('utf-8')
    anchor_source_code = base64.b64decode(dataset_submissions.loc[s['AcceptedAnchor']]['sourceCode']).decode('utf-8')

    oficial_tests = problem_data['official_tests']

    if not problem_data['official_tests_complete']:
        if problem_data['generated_tests'] > 0:
            for idx in generated_tests_ds_index_mapping[s['problem_id']]:
                test_data = generated_tests_ds[idx]
                oficial_tests.append({"input": test_data['input'], "output": test_data['output']})
    
    def check(response):
        return all([result and 'compile' in result and result['compile']['code'] == 0 and result['run']['code'] == 0 and result['run']['stdout'].split()[0] == '1' for result in response])
    
    problem_name = s['problem_id'].replace('/', '_')

    result_json = {"model_vendor": VENDOR,
                   "model_name": MODEL_NAME,
                   "timestamp": str(datetime.datetime.now()),
                   "problem_name": problem_name}
    
    root_path = os.path.join('results', problem_name, VENDOR, MODEL_NAME)
    log_path = os.path.join(root_path, 'log')

    if os.path.exists(os.path.join(root_path, f'{s.name}.json')):
        continue

    os.makedirs(root_path, exist_ok=True)
    os.makedirs(log_path, exist_ok=True)

    current_try = 0
    while current_try < MAX_TRIES:
        try:
            response_anchor = await compile_and_test(anchor_source_code, problem_data, oficial_tests, "http://host.docker.internal:2000", programmingLanguage)
            r_anchor = check(response_anchor)
            break
        except:
            current_try+=1
    if current_try >= MAX_TRIES:
        print(f'Something went wrong! Skipping {s.name}')
        continue

    result_json['anchor'] = {"id": f"{s['AcceptedAnchor']}_anchor", 
                            "source_code": anchor_source_code,
                            "result": r_anchor}
    
    current_try = 0
    while current_try < MAX_TRIES:
        try:
            response_wrong = await compile_and_test(source_code, problem_data, oficial_tests, "http://host.docker.internal:2000", programmingLanguage)
            r_wrong = check(response_wrong)
            break
        except:
            current_try+=1
    
    if current_try >= MAX_TRIES:
        print(f'Something went wrong! Skipping {s.name}')
        continue

    result_json['original'] = {"id": f'{s.name}_org',
                               "source_code": source_code,
                               "result": r_wrong}

    current_try = 0

    while current_try < MAX_TRIES:
        try:
            prompt = read_prompt('prompt_naive_fix_bug',
                                problem_description = problem_data['description'],
                                problem_input_format = problem_data['input_format'],
                                problem_output_format = problem_data['output_format'],
                                problem_example = str(problem_data['examples']),
                                note = problem_data['note'] if problem_data['note'] else "No Note",
                                submission_verdict = s['verdict'],
                                submission_code = source_code)
            
            response_generated = model_api.generate(MODEL_NAME, prompt)
            response_fix_code = extract_code(response_generated)
            response_fix_gene_code = await compile_and_test(response_fix_code, problem_data, oficial_tests, "http://host.docker.internal:2000", programmingLanguage)
            r_gene_fix_code = check(response_fix_gene_code)

            result_json['llm_fix_code'] = {"id": f"{s.name}_gene_fix",
                                            "prompt": prompt,
                                            "response": response_generated,
                                            "source_code": response_fix_code,
                                            "result": r_gene_fix_code}

            break
        except:
            log_error_result = {"id": f"{s.name}_gene_fix",
                                "prompt": prompt,
                                "response": response_generated}
            
            with open(os.path.join(log_path, f'{str(datetime.datetime.now())}_gene_fix.json'), 'w', encoding='utf-8') as fp:
                json.dump(log_error_result, fp)

            print("Failed to generate something useful")
            current_try+=1
    
    if current_try >= MAX_TRIES:
        print(f"None of the generated answers worked for submission {s.name}")
        continue

    current_try = 0
    while current_try < MAX_TRIES:
        try:
            prompt = read_prompt('prompt_generate_solution',
                                problem_description = problem_data['description'],
                                problem_input_format = problem_data['input_format'],
                                problem_output_format = problem_data['output_format'],
                                problem_example = str(problem_data['examples']),
                                note = problem_data['note'] if problem_data['note'] else "No Note")
            
            response_generated = model_api.generate(MODEL_NAME, prompt)
            response_code = extract_code(response_generated)

            response_gene_code = await compile_and_test(response_code, problem_data, oficial_tests, "http://host.docker.internal:2000", programmingLanguage)
            r_gene_code = check(response_gene_code)

            result_json['llm_generate_code'] = {"id": f"{s.name}_generate",
                                                "prompt": prompt,
                                                "response": response_generated,
                                                "source_code": response_code,
                                                "result": r_gene_code
                                               }
            break
        except:
            log_error_result = {"id": f"{s.name}_gene",
                                "prompt": prompt,
                                "response": response_generated}
            
            with open(os.path.join(log_path, f'{str(datetime.datetime.now())}_gene.json'), 'w', encoding='utf-8') as fp:
                json.dump(log_error_result, fp)

            print("Failed to generate something useful")
            current_try+=1
    
    if current_try >= MAX_TRIES:
        print(f"None of the generated answers worked for submission {s.name}")
        continue

    print(f"original ({s['verdict']}) -> judge anchor ({'OK' if r_anchor == True else 'wrong/TLE/MLE'}) -> judge wrong ({'OK' if r_wrong== True else 'wrong/TLE/MLE'}) -> gen fix code ({'OK' if r_gene_fix_code== True else 'wrong/TLE/MLE'}) -> gen code ({'OK' if r_gene_code== True else 'wrong/TLE/MLE'})")

    with open(os.path.join(root_path, f'{s.name}.json'), 'w', encoding='utf-8') as fp:
        json.dump(result_json, fp)

 40%|████      | 545/1357 [05:12<07:45,  1.74it/s]


NameError: name 'response_generated' is not defined